<a href="https://colab.research.google.com/github/fbd2003/class-05-forms/blob/main/Week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%config InlineBackend.figure_formats = ["retina"]

#Week 8: Tracking donated organs
This week we'll use a reduced version of a fairly sensitive dataset to understand several decades of history in the flow of donated livers, kidneys, and pancreases.

We went through a [data request process](https://optn.transplant.hrsa.gov/data/view-data-reports/request-data/data-request-instructions/) with the Organ Procurement and Transplantation Network (OPTN) to get a robust dataset based on STAR (Standard Transplant Analysis and Research) files.  Because lots of personal information was added that's potentially identifying, we're being very careful with the files.  You can access the prepared data through Canvas for use with this notebook, and we'll ask that you deestroy it at the end of the semester.

In [3]:
!pip install -q geopy seaborn statsmodels

In [4]:
import sqlite3

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from functools import partial

from geopy import distance
from shapely.geometry import Point

In [6]:
# you will have to download the sqlite dataset from the secure location and point at the right path here

optn_df = pd.read_sql("SELECT * FROM optn", sqlite3.connect("/content/optn_reduced.sqlite3"))
optn_df["deceased_donor"] = optn_df["deceased_donor"].astype(bool)
optn_df["succeeded_1_year"] = optn_df["succeeded_1_year"].astype(bool)

DatabaseError: Execution failed on sql 'SELECT * FROM optn': no such table: optn

In [ ]:
optn_df

This block defines distance between the hospitals.  Since it's based on latitude and longitude, this is distance "as the crow flies."

In [ ]:
def transplant_distance_miles(row):
    return distance.distance(
        (row["donor_lat"], row["donor_lon"]),
        (row["transplant_lat"], row["transplant_lon"])
    ).miles

In [ ]:
optn_df["distance_miles"] = optn_df.apply(transplant_distance_miles, axis=1)

In [ ]:
optn_df

In [ ]:
organs = ["Liver", "Kidney"]

How many of each kind of organ do we have in the data?

In [ ]:
optn_df["organ"].value_counts()

For organs that were moved (distance>0), let's make a histogram to see the distribution of the **cold ischemic time** (time out of the body and on ice).  

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
for organ in organs:
    optn_df[(optn_df["distance_miles"] > 0) & (optn_df["organ"] == organ.lower())].hist(
        "preservation_hours", ax=ax, alpha=0.8, bins=range(0, 49), density=True, label=organ,
    )

ax.set_title("Cold ischemic time by organ")
ax.set_xlabel("Cold ischemic time (hours)")
ax.set_ylabel("Freq")
plt.legend()
plt.show()

So it looks like kidneys are on ice longer.  Does that mean they travel farther?

In [ ]:
for organ in organs:
    fig, ax = plt.subplots(figsize=(8, 4))
    optn_df[(optn_df["distance_miles"] > 0) & (optn_df["organ"] == organ.lower())].hist(
        "distance_miles", ax=ax, bins=range(0, 2500, 50),density=True, label=organ,
    )

    ax.set_title(f"{organ} transit distance")
    ax.set_xlabel("Distance (miles)")
    ax.set_ylabel("Freq")
    plt.show()

Maybe not.  OK so there must be a complicated relationship between cold time and transit distance.

In [ ]:
for organ in organs:
    fig, ax = plt.subplots(figsize=(8, 4))
    optn_df[(optn_df["distance_miles"] > 0) & (optn_df["organ"] == organ.lower())].plot.scatter(
        y="preservation_hours", x="distance_miles", ax=ax, s=0.2,
    )
    ax.set_title(f"{organ} cold ischemic time vs. transit distance")
    ax.set_xlabel("Distance (miles)")
    ax.set_ylabel("Cold ischemic time (hours)")
    plt.show()

Those scatterplots have lots of interesting structure.  Now let's visualize the same data another way, using what's called a "KDE plot" (kernel density estimate).  This just takes the point data and smooths it out to see where the data is densest.

In [ ]:
for organ in organs:
    joint_df = optn_df[
        (optn_df["distance_miles"] > 0) &
        (optn_df["organ"] == organ.lower()) &
        (optn_df["distance_miles"] <= 2000)
    ]
    ax = sns.kdeplot(
        data=joint_df,
        x="distance_miles",
        y="preservation_hours",
        fill=True,
    )
    ax.set_title(f"{organ} cold ischemic time vs. transit distance")
    ax.set_xlabel("Distance (miles)")
    ax.set_ylabel("Cold ischemic time (hours)")
    plt.show()

Next, we'll use a technique called **logistic regression**.  This is a statistical technique for taking a bunch of data points of success and failure and fitting an S-shaped curve to the data.  The interpretation is that this is telling you an inferred PROBABILITY, learned from the data points.

In [ ]:
for organ in organs:
    success_df = optn_df[
        (optn_df["distance_miles"] > 0) &
        (optn_df["organ"] == organ.lower())
    ].sample(frac=0.2)
    ax = sns.regplot(
        data=success_df,
        x=success_df["distance_miles"],
        y=success_df["succeeded_1_year"],
        logistic=True,
    )
    ax.set_title(f"{organ} p(success) by transit distance")
    ax.set_xlabel("Distance (miles)")
    ax.set_ylabel("p(success)")
    plt.show()

In [ ]:
for organ in organs:
    success_df = optn_df[(optn_df["organ"] == organ.lower())].sample(frac=0.2)
    ax = sns.regplot(
        data=success_df,
        x=success_df["preservation_hours"],
        y=success_df["succeeded_1_year"],
        logistic=True,
    )
    ax.set_title(f"{organ} p(success) by cold ischemic time")
    ax.set_xlabel("Cold ischemic time (hours)")
    ax.set_ylabel("p(success)")
    plt.show()

## Transplant volume by Census region/division

Everything we did above was using geospatial data basically just to see how far organs are shipped.  In this section we'll pay attention to where they are sent FROM and TO.

We'll use 4 census regions (Northeast, Midwest, South, and West) and we'll also subdivide those a little more finely into nine "[divisions](https://www2.census.gov/geo/pdfs/maps-data/maps/reference/us_regdiv.pdf)," to get a feel for how organs move around the country.

In [ ]:
region_gdf = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_region_500k.zip")

In [ ]:
division_gdf = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_division_500k.zip")

In [ ]:
def loc_to_label(row, label_gdf, prefix, cache=None):
    coords = (row[f"{prefix}_lon"], row[f"{prefix}_lat"])
    if cache and coords in cache:
        return cache[coords]

    p = Point(*coords)
    matches = label_gdf.iloc[label_gdf.sindex.query(p, predicate="intersects")]
    if not len(matches):
        return "N/A"

    assert len(matches) == 1
    cache[coords] = matches.iloc[0]["NAME"]
    return matches.iloc[0]["NAME"]

In [ ]:
region_cache = {}
optn_df["transplant_census_region"] = optn_df.apply(
    partial(loc_to_label, label_gdf=region_gdf, prefix="transplant", cache=region_cache),
    axis=1,
)
optn_df["donor_census_region"] = optn_df.apply(
    partial(loc_to_label, label_gdf=region_gdf, prefix="donor", cache=region_cache),
    axis=1,
)

In [ ]:
division_cache = {}
optn_df["transplant_census_division"] = optn_df.apply(
    partial(loc_to_label, label_gdf=division_gdf, prefix="transplant", cache=division_cache),
    axis=1,
)
optn_df["donor_census_division"] = optn_df.apply(
    partial(loc_to_label, label_gdf=division_gdf, prefix="donor", cache=division_cache),
    axis=1,
)

OK so all that info is packed into our dataframe.  We've got columns for how long the organ was preserved, for the donor and recipient age (chunked as 0-29, 30-59, or 60+ years old) and we'll show whether the graft was marked successful after one year.

(Later, if you play around with more of the data in the sqlite file, you'll see columns for success to 6 months, 2 years, etc.)

In [ ]:
optn_df

In [ ]:
pd.crosstab(optn_df["transplant_census_region"], optn_df["donor_census_region"])

In [ ]:
pd.crosstab(optn_df["transplant_census_division"], optn_df["donor_census_division"])

# Homework 7 -- suggested due date Friday March 21 -- extended due date Tuesday March 25, 1:25pm

**Warmup question**: choose one census division from the table above, and choose one hospital in that division.  Figure out what city that hospital is in (reverse geocoding!).  You can do this by actually writing the lat, long coordinates in the Google Maps search bar, for example!  For that hospital, how many liver transplants did it receive in this whole dataset?  Explain how you figured it out.

**Data product**: as usual, use this notebook to get curious about some aspect of organ donation flows.  Formulate a question and make a data product that addresses it.  Briefly explain how you made it.

**Reading question**: This week's reading is by Kieran Healy on the sociology of organ donation.  He spends a good deal of space in the chapter discussing the ways that organ donation falls into a public policy gray area---is it best handled as a *gift* or by a *market*?  Pick out one quote from the chapter that relates to this tension between virtue and incentives.  (Cite the page your quote is on.)  Then give another example, not already covered in the chapter or the class discussion, of a policy question that falls in a gray area between gifts and markets.  Be creative!

In [ ]:
# prompt: choose one census division from the table above, and choose one hospital in that division. Figure out what city that hospital is in (reverse geocoding!). You can do this by actually writing the lat, long coordinates in the Google Maps search bar, for example! For that hospital, how many liver transplants did it receive in this whole dataset? Explain how you figured it out

# Choose a census division (e.g., East South Central)
division = "East South Central"

# Filter the DataFrame to get hospitals in that division
hospitals_in_division = optn_df[optn_df["transplant_census_division"] == division]

# Choose one hospital from that division (e.g., the first one)
hospital = hospitals_in_division.iloc[0]

# Extract the latitude and longitude of the hospital
transplant_lat = hospital["transplant_lat"]
transplant_lon = hospital["transplant_lon"]

# Use Google Maps or a similar service to reverse geocode the coordinates
# Example: Searching "33.4484, -86.7973" (lat/long of Birmingham, AL) on Google Maps
# reveals the city as Birmingham, Alabama

# Now, let's find the number of liver transplants for that hospital
liver_transplants_at_hospital = optn_df[
    (optn_df["transplant_lat"] == transplant_lat) &
    (optn_df["transplant_lon"] == transplant_lon) &
    (optn_df["organ"] == "liver")
].shape[0]

print(f"The hospital at coordinates ({transplant_lat}, {transplant_lon}) received {liver_transplants_at_hospital} liver transplants.")

# Explanation:
# 1. We filtered the DataFrame to get all rows that match the latitude and longitude of the chosen hospital.
# 2. We further filtered the DataFrame to only include rows where the organ transplanted was a liver.
# 3. We used the shape[0] to count the number of rows remaining after the filters, indicating the number of liver transplants at that specific hospital.



In [ ]:
# prompt: use this notebook to get curious about some aspect of organ donation flows. Formulate a question and make a data product that addresses it. Briefly explain how you made it.

import pandas as pd
import matplotlib.pyplot as plt
# Question: How does the distance between donor and recipient hospitals affect the likelihood of a successful liver transplant?

# Data Product: A visualization showing the relationship between transplant distance and the success rate of liver transplants.


# 1. Filter the data for liver transplants only
liver_df = optn_df[optn_df["organ"] == "liver"]

# 2. Group the data by distance and calculate the success rate for each distance range.
distance_bins = [0, 100, 500, 1000, 2000, float('inf')]  # Define distance ranges
liver_df['distance_range'] = pd.cut(liver_df['distance_miles'], bins=distance_bins)

success_rates = liver_df.groupby('distance_range')['succeeded_1_year'].mean()

# 3. Create a bar chart to visualize the relationship
plt.figure(figsize=(10, 6))
plt.bar(success_rates.index.astype(str), success_rates.values)
plt.xlabel('Distance Range (miles)')
plt.ylabel('Success Rate (1-year)')
plt.title('Success Rate of Liver Transplants by Distance')
plt.xticks(rotation=45, ha='right')
plt.show()


# Explanation:
#
# 1. We filtered the original DataFrame to only include data for liver transplants.
# 2. We grouped the data by distance ranges and calculated the average success rate within each range.
# 3. We created a bar chart to visualize the success rate for each distance range, showing how the likelihood of a successful liver transplant is associated with the distance between the donor and recipient hospitals.

# Further analysis could include:
# - Looking at the impact of other factors, such as donor age and recipient age, on the success rate.
# - Investigating whether there are geographic regions where transplant success rates are consistently higher or lower, potentially due to differences in resource availability or infrastructure.

